In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import *
from config.feature_registry import NASA_COLUMN_RENAME
from src.preprocessing.merger import DataMerger
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.aggregator import TemporalAggregator
from src.preprocessing.feature_engineer import FeatureEngineer
from src.preprocessing.scaler import FeatureScaler, DimensionalityReducer

plt.rcParams["figure.figsize"] = (14, 6)
sns.set_theme(style="whitegrid")
print("✅ Setup complete")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  LOAD COMBINED DATA
# ══════════════════════════════════════════════════════════════

om_df = pd.read_csv(INTERIM_DIR / "open_meteo_combined.csv")
om_df["date"] = pd.to_datetime(om_df["date"])

nasa_df = pd.read_csv(INTERIM_DIR / "nasa_power_combined.csv")
nasa_df["date"] = pd.to_datetime(nasa_df["date"])

print(f"🌤️  Open-Meteo:  {om_df.shape}")
print(f"🛰️  NASA POWER:  {nasa_df.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 1: MERGE OPEN-METEO + NASA POWER
# ══════════════════════════════════════════════════════════════

merger = DataMerger()
merged = merger.merge(om_df, nasa_df)

print(f"\n📊 Merged shape: {merged.shape}")

# Check merge quality
report = merger.quality_report(merged)
print(f"\n📋 Merge Quality Report:")
print(f"   Total rows:      {report['total_rows']:,}")
print(f"   Total columns:   {report['total_columns']}")
print(f"   Unique points:   {report['unique_points']}")
print(f"   Date range:      {report['date_range']}")
print(f"   Open-Meteo cols: {report['open_meteo_columns']}")
print(f"   NASA cols:       {report['nasa_columns']}")

# Check NASA match rate
nasa_cols = [c for c in merged.columns if c.startswith("nasa_")]
if nasa_cols:
    match_rate = merged[nasa_cols[0]].notna().mean() * 100
    print(f"   NASA match rate: {match_rate:.1f}%")

# Save merged
merger.save(merged)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 2: CLEAN DATA
# ══════════════════════════════════════════════════════════════

print("🔍 Before cleaning:")
print(f"   Shape: {merged.shape}")
print(f"   Total nulls: {merged.isnull().sum().sum():,}")

cleaner = DataCleaner(
    null_threshold=0.5,
    outlier_method="iqr",
    outlier_factor=3.0,
)
cleaned = cleaner.clean(merged)

print(f"\n🔍 After cleaning:")
print(f"   Shape: {cleaned.shape}")
print(f"   Total nulls: {cleaned.isnull().sum().sum():,}")

summary = cleaner.summary(cleaned)
print(f"\n📋 Cleaning Summary:")
for k, v in summary.items():
    print(f"   {k}: {v}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  VISUALIZE CLEANING IMPACT
# ══════════════════════════════════════════════════════════════

compare_cols = ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_mean"]
available_compare = [c for c in compare_cols if c in merged.columns and c in cleaned.columns]

if available_compare:
    fig, axes = plt.subplots(len(available_compare), 2, figsize=(14, 4 * len(available_compare)))
    if len(available_compare) == 1:
        axes = [axes]

    for idx, col in enumerate(available_compare):
        # Before
        axes[idx][0].hist(merged[col].dropna(), bins=60, color="salmon", alpha=0.7, edgecolor="white")
        axes[idx][0].set_title(f"BEFORE: {col}", fontsize=10, fontweight="bold")
        axes[idx][0].axvline(merged[col].mean(), color="red", linestyle="--")

        # After
        axes[idx][1].hist(cleaned[col].dropna(), bins=60, color="steelblue", alpha=0.7, edgecolor="white")
        axes[idx][1].set_title(f"AFTER: {col}", fontsize=10, fontweight="bold")
        axes[idx][1].axvline(cleaned[col].mean(), color="red", linestyle="--")

    plt.suptitle("Data Cleaning: Before vs After", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / "preprocessing_cleaning_impact.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 3: FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════

engineer = FeatureEngineer()
featured = engineer.transform(cleaned)

# What new features were added?
new_cols = set(featured.columns) - set(cleaned.columns)
print(f"📊 Before engineering: {cleaned.shape[1]} columns")
print(f"📊 After engineering:  {featured.shape[1]} columns")
print(f"📊 New features added: {len(new_cols)}")
print(f"\n📋 New Features:")
for col in sorted(new_cols):
    sample = featured[col].dropna().head(3).tolist()
    print(f"   ✅ {col:<40s} sample: {[round(v, 2) for v in sample]}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SAVE DAILY PROCESSED DATA
# ══════════════════════════════════════════════════════════════

featured.to_csv(PROCESSED_DIR / "feature_matrix_daily.csv", index=False)
size_mb = (PROCESSED_DIR / "feature_matrix_daily.csv").stat().st_size / (1024 ** 2)
print(f"💾 Saved: feature_matrix_daily.csv ({len(featured):,} rows, {size_mb:.1f} MB)")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 4: TEMPORAL AGGREGATION
# ══════════════════════════════════════════════════════════════

aggregator = TemporalAggregator()

# Monthly
monthly = aggregator.to_monthly(featured)
monthly.to_csv(PROCESSED_DIR / "feature_matrix_monthly.csv", index=False)
print(f"📅 Monthly:   {monthly.shape}")

# Seasonal
seasonal = aggregator.to_seasonal(featured)
seasonal.to_csv(PROCESSED_DIR / "feature_matrix_seasonal.csv", index=False)
print(f"🌦️  Seasonal:  {seasonal.shape}")

# Yearly
yearly = aggregator.to_yearly(featured)
yearly.to_csv(PROCESSED_DIR / "feature_matrix_yearly.csv", index=False)
print(f"📆 Yearly:    {yearly.shape}")

# Climate Normals (THE KEY OUTPUT for clustering)
normals = aggregator.to_climate_normals(featured)
normals = engineer.transform(normals)  # Add derived features to normals too
normals.to_csv(PROCESSED_DIR / "climate_normals.csv", index=False)
print(f"🌍 Normals:   {normals.shape}")

print(f"\n📊 Climate normals — one row per grid point:")
print(f"   Points:   {len(normals)}")
print(f"   Features: {len(normals.columns)}")
print(normals.head())

In [ ]:
# ══════════════════════════════════════════════════════════════
#  VISUALIZE DIFFERENT AGGREGATION LEVELS
# ══════════════════════════════════════════════════════════════

# Pick one point for demonstration
sample_lat = normals["latitude"].iloc[len(normals) // 2]
sample_lon = normals["longitude"].iloc[len(normals) // 2]
print(f"📍 Sample point: ({sample_lat}, {sample_lon})")

fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=False)

col = "temperature_2m_mean" if "temperature_2m_mean" in featured.columns else featured.select_dtypes(include=[np.number]).columns[0]

# Daily
daily_pt = featured[
    (featured["latitude"] == sample_lat) & (featured["longitude"] == sample_lon)
].sort_values("date")
if len(daily_pt) > 0:
    axes[0].plot(pd.to_datetime(daily_pt["date"]), daily_pt[col], linewidth=0.3, alpha=0.7)
    axes[0].set_title(f"Daily — {col}", fontweight="bold")
    axes[0].grid(True, alpha=0.3)

# Monthly
monthly_pt = monthly[
    (monthly["latitude"] == sample_lat) & (monthly["longitude"] == sample_lon)
]
if len(monthly_pt) > 0:
    monthly_pt = monthly_pt.sort_values(["year", "month"])
    x = range(len(monthly_pt))
    axes[1].plot(x, monthly_pt[col], "o-", markersize=2)
    axes[1].set_title(f"Monthly — {col}", fontweight="bold")
    axes[1].grid(True, alpha=0.3)

# Seasonal
seasonal_pt = seasonal[
    (seasonal["latitude"] == sample_lat) & (seasonal["longitude"] == sample_lon)
]
if len(seasonal_pt) > 0:
    seasonal_pt = seasonal_pt.sort_values("season_year")
    season_colors = {"Winter": "blue", "Pre-Monsoon": "orange", "Monsoon": "green", "Post-Monsoon": "red"}
    for season, color in season_colors.items():
        s_data = seasonal_pt[seasonal_pt["season"] == season]
        axes[2].plot(s_data["season_year"], s_data[col], "o-", color=color, label=season, markersize=3)
    axes[2].set_title(f"Seasonal — {col}", fontweight="bold")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

# Yearly
yearly_pt = yearly[
    (yearly["latitude"] == sample_lat) & (yearly["longitude"] == sample_lon)
].sort_values("year")
if len(yearly_pt) > 0:
    axes[3].plot(yearly_pt["year"], yearly_pt[col], "o-", color="purple", markersize=5)
    axes[3].set_title(f"Yearly — {col}", fontweight="bold")
    axes[3].grid(True, alpha=0.3)

plt.suptitle(f"Aggregation Levels — Point ({sample_lat}, {sample_lon})", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "preprocessing_aggregation_levels.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 5: SCALING PREVIEW (for clustering)
# ══════════════════════════════════════════════════════════════

exclude = ["latitude", "longitude", "date", "year", "month", "season", "days_count"]
feature_cols = [
    c for c in normals.select_dtypes(include=[np.number]).columns
    if c not in exclude
]

print(f"📋 Features for clustering: {len(feature_cols)}")

# Compare scalers
scalers = {
    "Standard": FeatureScaler(method="standard"),
    "MinMax": FeatureScaler(method="minmax"),
    "Robust": FeatureScaler(method="robust"),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, scaler) in zip(axes, scalers.items()):
    X = scaler.fit_transform(normals, feature_cols)

    # Show distribution of first 5 features
    for i in range(min(5, X.shape[1])):
        ax.hist(X[:, i], bins=30, alpha=0.4, label=feature_cols[i][:20])

    ax.set_title(f"{name} Scaler", fontsize=12, fontweight="bold")
    ax.set_xlabel("Scaled Value")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Scaler Comparison (First 5 Features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "preprocessing_scaler_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 6: PCA — DIMENSIONALITY REDUCTION
# ══════════════════════════════════════════════════════════════

scaler = FeatureScaler(method="standard")
X_scaled = scaler.fit_transform(normals, feature_cols)

reducer = DimensionalityReducer()
X_pca = reducer.fit_pca(X_scaled, n_components=0.95, feature_names=feature_cols)

# Variance explained
variance = reducer.get_explained_variance()
cumulative = np.cumsum(variance)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual
ax1.bar(range(1, len(variance) + 1), variance * 100, color="steelblue", alpha=0.8)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Variance Explained (%)")
ax1.set_title("Individual Variance per Component", fontweight="bold")
ax1.grid(True, alpha=0.3)

# Cumulative
ax2.plot(range(1, len(cumulative) + 1), cumulative * 100, "ro-", markersize=5)
ax2.axhline(y=95, color="gray", linestyle="--", label="95% threshold")
ax2.fill_between(range(1, len(cumulative) + 1), cumulative * 100, alpha=0.2, color="red")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance (%)")
ax2.set_title("Cumulative Variance Explained", fontweight="bold")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(
    f"PCA: {X_scaled.shape[1]} features → {reducer.n_components} components (95% variance)",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "preprocessing_pca.png"), dpi=150, bbox_inches="tight")
plt.show()

# 2D scatter of PCA
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=normals["latitude"], cmap="RdYlBu_r",
    s=20, alpha=0.7,
)
plt.colorbar(scatter, ax=ax, label="Latitude")
ax.set_xlabel(f"PC1 ({variance[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({variance[1]*100:.1f}%)")
ax.set_title("PCA 2D — Colored by Latitude", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "preprocessing_pca_2d.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
#  PREPROCESSING SUMMARY
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("  📋 PREPROCESSING COMPLETE — SUMMARY")
print("=" * 70)
print(f"""
  📁 Files Created:
     {PROCESSED_DIR / 'feature_matrix_daily.csv'}
     {PROCESSED_DIR / 'feature_matrix_monthly.csv'}
     {PROCESSED_DIR / 'feature_matrix_seasonal.csv'}
     {PROCESSED_DIR / 'feature_matrix_yearly.csv'}
     {PROCESSED_DIR / 'climate_normals.csv'}  ← MAIN CLUSTERING INPUT
     
  📊 Data Flow:
     Raw Open-Meteo:     {om_df.shape}
     Raw NASA POWER:     {nasa_df.shape}
     Merged:             {merged.shape}
     Cleaned:            {cleaned.shape}
     Feature engineered: {featured.shape}
     
  📊 Aggregation Outputs:
     Daily:    {featured.shape}
     Monthly:  {monthly.shape}
     Seasonal: {seasonal.shape}
     Yearly:   {yearly.shape}
     Normals:  {normals.shape}  ← 1 row per grid point
     
  📊 Clustering Input Ready:
     Points:     {len(normals)}
     Features:   {len(feature_cols)}
     PCA dims:   {reducer.n_components} (95% variance retained)
     
  ✅ Ready for notebook 05_clustering_experiments!
""")
print("=" * 70)